# Operating Systems Lab: Page Replacement Algorithm Simulator

**Algorithms:** FIFO, LRU, Optimal  
**Purpose:** Simulate page replacement, display step-by-step frame contents, calculate hits/faults and compare performance.

> Before submission, run all cells in Google Colab, review the outputs, and add your name/roll number. Make sure you understand the code and follow your instructor's rules about individual work.

## 1. Input

Enter a page reference string separated by spaces or commas, followed by the number of frames.

In [ ]:
def parse_reference_string(raw):
    """Parse comma- or space-separated page references into integers."""
    parts = raw.replace(",", " ").split()
    if not parts:
        raise ValueError("Please enter at least one page number.")

    try:
        pages = [int(part) for part in parts]
    except ValueError:
        raise ValueError("Invalid reference string: use numeric page values only.")

    return pages


def get_user_input():
    """Read and validate the reference string and frame count."""
    pages = parse_reference_string(input("Enter page reference string (spaces or commas): "))

    try:
        num_frames = int(input("Enter number of frames: "))
    except ValueError:
        raise ValueError("Invalid frame count: enter a whole number.")

    if num_frames < 1:
        raise ValueError("Number of frames must be at least 1.")

    return pages, num_frames


# Run this cell to enter your own test case.
reference_string, frame_count = get_user_input()
print("\nReference string:", reference_string)
print("Number of frames:", frame_count)

Enter page reference string (spaces or commas): 1 2 3 1 4 2 5
Enter number of frames: 3

Reference string: [1, 2, 3, 1, 4, 2, 5]
Number of frames: 3


## 2. FIFO — First-In, First-Out

FIFO replaces the page that has been in memory the longest. The order of arrival is maintained using a queue-like list.

In [ ]:
def fifo(reference, num_frames):
    frames = []
    next_to_replace = 0
    faults = 0
    hits = 0
    trace = []

    for step, page in enumerate(reference, start=1):
        replaced = "-"
        if page in frames:
            status = "Hit"
            hits += 1
        else:
            status = "Fault"
            faults += 1

            if len(frames) < num_frames:
                frames.append(page)
            else:
                replaced = frames[next_to_replace]
                frames[next_to_replace] = page
                next_to_replace = (next_to_replace + 1) % num_frames

        trace.append({
            "Step": step,
            "Page": page,
            "Frames": frames.copy(),
            "Status": status,
            "Replaced": replaced
        })

    return faults, hits, trace

## 3. LRU — Least Recently Used

LRU replaces the page that has not been used for the longest time in the past.

In [ ]:
def lru(reference, num_frames):
    frames = []
    last_used = {}
    faults = 0
    hits = 0
    trace = []

    for step, page in enumerate(reference, start=1):
        replaced = "-"
        if page in frames:
            status = "Hit"
            hits += 1
        else:
            status = "Fault"
            faults += 1

            if len(frames) < num_frames:
                frames.append(page)
            else:
                # Select the page whose most recent use was earliest.
                victim = min(frames, key=lambda p: last_used[p])
                replaced = victim
                frames[frames.index(victim)] = page

        last_used[page] = step
        trace.append({
            "Step": step,
            "Page": page,
            "Frames": frames.copy(),
            "Status": status,
            "Replaced": replaced
        })

    return faults, hits, trace

## 4. Optimal Page Replacement

Optimal replaces the page whose next use is farthest in the future. If multiple pages will never be used again, the page appearing first in the frames is replaced, as specified in the assignment.

In [ ]:
def optimal(reference, num_frames):
    frames = []
    faults = 0
    hits = 0
    trace = []

    for index, page in enumerate(reference):
        replaced = "-"
        if page in frames:
            status = "Hit"
            hits += 1
        else:
            status = "Fault"
            faults += 1

            if len(frames) < num_frames:
                frames.append(page)
            else:
                future = reference[index + 1:]
                next_use = {}

                for frame_page in frames:
                    try:
                        next_use[frame_page] = future.index(frame_page)
                    except ValueError:
                        next_use[frame_page] = float("inf")

                # max() returns the first frame in a tie because frames are traversed in order.
                farthest_distance = max(next_use.values())
                victim = next(p for p in frames if next_use[p] == farthest_distance)
                replaced = victim
                frames[frames.index(victim)] = page

        trace.append({
            "Step": index + 1,
            "Page": page,
            "Frames": frames.copy(),
            "Status": status,
            "Replaced": replaced
        })

    return faults, hits, trace

## 5. Display trace and performance metrics

Hit ratio = hits / total references. Fault ratio = faults / total references.

In [ ]:
def show_trace(name, faults, hits, trace, total_pages):
    print(f"\\n{'=' * 78}")
    print(name)
    print("=" * 78)
    print(f"{'Step':<6}{'Page':<8}{'Frame contents':<28}{'Result':<10}{'Replaced':<10}")
    print("-" * 78)

    for row in trace:
        frame_text = "[" + ", ".join(map(str, row["Frames"])) + "]"
        print(f"{row['Step']:<6}{row['Page']:<8}{frame_text:<28}"
              f"{row['Status']:<10}{str(row['Replaced']):<10}")

    hit_ratio = hits / total_pages if total_pages else 0
    fault_ratio = faults / total_pages if total_pages else 0
    print("-" * 78)
    print(f"Total page faults : {faults}")
    print(f"Total page hits   : {hits}")
    print(f"Hit ratio         : {hits}/{total_pages} = {hit_ratio:.2%}")
    print(f"Fault ratio       : {faults}/{total_pages} = {fault_ratio:.2%}")


def run_all_algorithms(reference, num_frames):
    if not reference:
        raise ValueError("Reference string must not be empty.")
    if not isinstance(num_frames, int) or num_frames < 1:
        raise ValueError("Number of frames must be a positive integer.")

    results = {}
    algorithms = [
        ("FIFO", fifo),
        ("LRU", lru),
        ("Optimal", optimal)
    ]

    for name, algorithm in algorithms:
        faults, hits, trace = algorithm(reference, num_frames)
        results[name] = {
            "faults": faults,
            "hits": hits,
            "hit_ratio": hits / len(reference),
            "fault_ratio": faults / len(reference),
            "trace": trace
        }
        show_trace(name, faults, hits, trace, len(reference))

    print("\n" + "=" * 78)
    print("FINAL COMPARISON")
    print("=" * 78)
    print(f"{'Algorithm':<12}{'Faults':<10}{'Hits':<10}{'Hit ratio':<14}{'Fault ratio':<14}")
    print("-" * 60)
    for name in ("FIFO", "LRU", "Optimal"):
        r = results[name]
        print(f"{name:<12}{r['faults']:<10}{r['hits']:<10}"
              f"{r['hit_ratio']:<14.2%}{r['fault_ratio']:<14.2%}")

    return results


results = run_all_algorithms(reference_string, frame_count)

\n==============================================================================
FIFO
Step  Page    Frame contents              Result    Replaced  
------------------------------------------------------------------------------
1     1       [1]                         Fault     -         
2     2       [1, 2]                      Fault     -         
3     3       [1, 2, 3]                   Fault     -         
4     1       [1, 2, 3]                   Hit       -         
5     4       [4, 2, 3]                   Fault     1         
6     2       [4, 2, 3]                   Hit       -         
7     5       [4, 5, 3]                   Fault     2         
------------------------------------------------------------------------------
Total page faults : 5
Total page hits   : 2
Hit ratio         : 2/7 = 28.57%
Fault ratio       : 5/7 = 71.43%
\n==============================================================================
LRU
Step  Page    Frame contents              Result    Repla

## 6. Sample test

Expected sample input from the assignment:
- Reference string: `1 2 3 1 4 2 5`
- Number of frames: `3`

The following cell runs this example independently of your interactive input.

In [ ]:
sample_reference = [1, 2, 3, 1, 4, 2, 5]
sample_frames = 3
sample_results = run_all_algorithms(sample_reference, sample_frames)

\n==============================================================================
FIFO
Step  Page    Frame contents              Result    Replaced  
------------------------------------------------------------------------------
1     1       [1]                         Fault     -         
2     2       [1, 2]                      Fault     -         
3     3       [1, 2, 3]                   Fault     -         
4     1       [1, 2, 3]                   Hit       -         
5     4       [4, 2, 3]                   Fault     1         
6     2       [4, 2, 3]                   Hit       -         
7     5       [4, 5, 3]                   Fault     2         
------------------------------------------------------------------------------
Total page faults : 5
Total page hits   : 2
Hit ratio         : 2/7 = 28.57%
Fault ratio       : 5/7 = 71.43%
\n==============================================================================
LRU
Step  Page    Frame contents              Result    Repla

# Part B — Manual Tracing

Reference string: `7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2`  
Frames: `3`

Run the following cell to generate the exact trace tables for FIFO, LRU, and Optimal. Use these tables to verify your handwritten manual tracing.

In [ ]:
part_b_reference = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]
part_b_frames = 3
part_b_results = run_all_algorithms(part_b_reference, part_b_frames)

\n==============================================================================
FIFO
Step  Page    Frame contents              Result    Replaced  
------------------------------------------------------------------------------
1     7       [7]                         Fault     -         
2     0       [7, 0]                      Fault     -         
3     1       [7, 0, 1]                   Fault     -         
4     2       [2, 0, 1]                   Fault     7         
5     0       [2, 0, 1]                   Hit       -         
6     3       [2, 3, 1]                   Fault     0         
7     0       [2, 3, 0]                   Fault     1         
8     4       [4, 3, 0]                   Fault     2         
9     2       [4, 2, 0]                   Fault     3         
10    3       [4, 2, 3]                   Fault     0         
11    0       [0, 2, 3]                   Fault     4         
12    3       [0, 2, 3]                   Hit       -         
13    2       [0

# Part C — Experiment 1: Belady's Anomaly

Reference string: `1 2 3 4 1 2 5 1 2 3 4 5`

Compare FIFO, LRU, and Optimal with 3 and 4 frames. Belady's anomaly is the situation where increasing the number of frames increases page faults.

In [ ]:
belady_reference = [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]
belady_results = {}

for frames in (3, 4):
    print(f"\\nExperiment with {frames} frames")
    belady_results[frames] = {}
    for name, algorithm in (("FIFO", fifo), ("LRU", lru), ("Optimal", optimal)):
        faults, hits, trace = algorithm(belady_reference, frames)
        belady_results[frames][name] = {"faults": faults, "hits": hits}
        print(f"{name:<10} Faults: {faults:<3} Hits: {hits}")

print("\\nBelady's anomaly comparison")
print(f"{'Algorithm':<12}{'3 frames':<12}{'4 frames':<12}")
for name in ("FIFO", "LRU", "Optimal"):
    print(f"{name:<12}{belady_results[3][name]['faults']:<12}"
          f"{belady_results[4][name]['faults']:<12}")

\nExperiment with 3 frames
FIFO       Faults: 9   Hits: 3
LRU        Faults: 10  Hits: 2
Optimal    Faults: 7   Hits: 5
\nExperiment with 4 frames
FIFO       Faults: 10  Hits: 2
LRU        Faults: 8   Hits: 4
Optimal    Faults: 6   Hits: 6
\nBelady's anomaly comparison
Algorithm   3 frames    4 frames    
FIFO        9           10          
LRU         10          8           
Optimal     7           6           


## Part C — Short-answer guidance

1. **Why can Optimal not be implemented exactly in a real OS?** It requires knowing future page references, which an operating system cannot know in advance. It is mainly used as a theoretical benchmark.
2. **What extra bookkeeping does LRU need compared with FIFO?** LRU must track the recency of page use (for example, timestamps or an ordered structure). FIFO mainly tracks arrival order.
3. **If frames equal the number of distinct pages, how many faults occur?** Assuming all frames start empty and the frame count is at least the number of distinct pages, each distinct page faults on its first reference and is then kept in memory. Faults equal the number of distinct pages; subsequent references are hits.

## Part C — Experiment 2: Your own input

The cell below provides two example strings of at least 15 references. Run it and inspect the fault counts. If the results do not satisfy the requested comparison on your version, adjust the reference strings and rerun until you find one where LRU is better and one where FIFO is better than or equal to LRU. Record your own verified strings and counts in the report.

In [ ]:
custom_cases = {
    "LRU better candidate": [1, 2, 3, 1, 2, 4, 1, 2, 3, 1, 2, 4, 1, 2, 3, 1],
    "FIFO better/equal candidate": [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5, 1, 2, 3, 4]
}
custom_frame_count = 3

for case_name, pages in custom_cases.items():
    fifo_faults, _, _ = fifo(pages, custom_frame_count)
    lru_faults, _, _ = lru(pages, custom_frame_count)
    print(case_name)
    print("Reference string:", " ".join(map(str, pages)))
    print("Frames:", custom_frame_count)
    print("FIFO faults:", fifo_faults)
    print("LRU faults :", lru_faults)
    print()

LRU better candidate
Reference string: 1 2 3 1 2 4 1 2 3 1 2 4 1 2 3 1
Frames: 3
FIFO faults: 11
LRU faults : 7

FIFO better/equal candidate
Reference string: 1 2 3 4 1 2 5 1 2 3 4 5 1 2 3 4
Frames: 3
FIFO faults: 13
LRU faults : 14



## Conclusion

FIFO, LRU, and Optimal use different rules to select a page for replacement. The trace and performance metrics help compare their behaviour on the same reference string. Optimal provides a theoretical lower-bound benchmark for page faults, while FIFO and LRU are practical policies with different bookkeeping requirements.

## GitHub checklist

- [ ] Add your name and roll number to the notebook.
- [ ] Run all cells in Google Colab and ensure outputs are visible.
- [ ] Verify the manual trace and experiment results.
- [ ] Add screenshots of executed output to your repository.
- [ ] Include this notebook and a README in your GitHub repository.
- [ ] Cite any external references you used and follow your instructor's submission rules.